# MILK10k External Validation — Feature Extraction + Evaluation
Loads each fold's fine-tuned PanDerm checkpoint, extracts 1024-dim CLS features
from MILK10k dermoscopy images, aggregates to patient level, and evaluates
with logistic regression.

**Classes:** Nevus (0) | Melanoma Invasive (1) | Melanoma in situ (2)

**Run after:** prepare_data, segment_lesions, make_panderm_csv for MILK10k.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

## Cell 2 — Install dependencies + restart runtime
Run once. After restart skip this cell and go to Cell 3.

In [ ]:
import os
!pip install timm==0.9.16 open_clip_torch 'numpy<2.0' -q
os.kill(os.getpid(), 9)

## Cell 3 — Verify environment

In [ ]:
import timm, numpy as np, torch
print(f'timm   : {timm.__version__}  (need 0.9.16)')
print(f'numpy  : {np.__version__}   (need <2.0)')
print(f'torch  : {torch.__version__}')
print(f'GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')

# Clone PanDerm if not already present
import os
if not os.path.exists('/content/PanDerm'):
    !git clone https://github.com/SiyuanYan1/PanDerm.git /content/PanDerm
    print('PanDerm cloned')
else:
    print('PanDerm already present')

import sys
if '/content/PanDerm/classification' not in sys.path:
    sys.path.insert(0, '/content/PanDerm/classification')
print('PanDerm added to path')

## Cell 4 — Config

In [ ]:
from pathlib import Path

# ── Edit these paths to match your Drive layout ────────────────────────────────
DRIVE_ROOT      = Path('/content/drive/MyDrive/PanDerm_Scripts')
MILK10K_ROOT    = DRIVE_ROOT / 'external-validation/milk10k'

CONFIG = {
    'panderm_repo':    '/content/PanDerm/classification',
    'checkpoint':      str(DRIVE_ROOT / 'panderm_ll_data6_checkpoint-499.pth'),  # pre-trained checkpoint (base)
    'results_dir':     str(DRIVE_ROOT / 'results'),                              # internal fold checkpoints
    'manifest':        str(MILK10K_ROOT / 'milk10k_manifest.csv'),
    'segmented_cache': str(MILK10K_ROOT / 'segmented_cache'),
    'output_dir':      str(MILK10K_ROOT / 'features'),
    'model':           'PanDerm_Large_FT',
    'nb_classes':      3,
    'batch_size':      32,
    'num_workers':     0,
    'n_folds':         5,
}
# ───────────────────────────────────────────────────────────────────────────────

Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
print('CONFIG ready')
for k, v in CONFIG.items():
    print(f'  {k:<20}: {v}')

## Cell 5 — Patch torch.load + pre-flight checks

In [ ]:
import torch
from pathlib import Path
import pandas as pd

# Patch torch.load for PyTorch 2.x
_original_load = torch.load
def _patched_load(f, map_location=None, pickle_module=None, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(f, map_location=map_location, **kwargs)
torch.load = _patched_load
print('torch.load patched')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Check manifest
manifest = pd.read_csv(CONFIG['manifest'])
manifest['patient_id'] = manifest['patient_id'].astype(str)
print(f'Manifest loaded: {len(manifest)} images, {manifest["patient_id"].nunique()} patients')
print(manifest['diagnosis'].value_counts())

# Check fold checkpoints
print('\nChecking fold checkpoints:')
for fold_idx in range(CONFIG['n_folds']):
    ckpt = Path(CONFIG['results_dir']) / f'results_fold{fold_idx}' / 'checkpoint-best.pth'
    status = 'OK' if ckpt.exists() else 'MISSING'
    print(f'  Fold {fold_idx}: {status} — {ckpt}')

## Cell 6 — Feature extraction (GPU)
Loads each fold's fine-tuned checkpoint and extracts 1024-dim CLS features
from all 1182 MILK10k images. Saves one feature file per fold.

In [ ]:
import sys, torch, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

sys.path.insert(0, CONFIG['panderm_repo'])
from models.PanDerm import panderm_large_patch16_224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.228, 0.224, 0.225]  # PanDerm-specific

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class MILK10kDataset(Dataset):
    def __init__(self, manifest_df, segmented_cache, transform):
        self.df        = manifest_df.reset_index(drop=True)
        self.seg_cache = Path(segmented_cache)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        seg_path = Path(row['segmented_path'])
        img_path = Path(row['image_path'])

        path = seg_path if seg_path.exists() else img_path
        img  = Image.open(str(path)).convert('RGB')
        return self.transform(img), int(row['label']), row['patient_id']


def load_panderm_encoder(checkpoint_path, nb_classes, device):
    model = panderm_large_patch16_224(num_classes=nb_classes)
    state = torch.load(checkpoint_path, map_location='cpu')
    ckpt  = state.get('model', state)
    missing, unexpected = model.load_state_dict(ckpt, strict=False)
    print(f'    missing={len(missing)}, unexpected={len(unexpected)}')
    model.eval()
    return model.to(device)


OUT_DIR  = Path(CONFIG['output_dir'])
n_folds  = CONFIG['n_folds']
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = MILK10kDataset(manifest, CONFIG['segmented_cache'], transform)
loader  = DataLoader(dataset, batch_size=CONFIG['batch_size'],
                     shuffle=False, num_workers=CONFIG['num_workers'])

all_labels     = np.array([int(manifest.iloc[i]['label']) for i in range(len(manifest))])
all_patient_ids = manifest['patient_id'].values
all_folds       = manifest['fold'].values

for fold_idx in range(n_folds):
    out_path = OUT_DIR / f'milk10k_features_fold{fold_idx}.npy'
    if out_path.exists():
        print(f'Fold {fold_idx}: already done, skipping')
        continue

    ckpt_path = Path(CONFIG['results_dir']) / f'results_fold{fold_idx}' / 'checkpoint-best.pth'
    if not ckpt_path.exists():
        print(f'Fold {fold_idx}: checkpoint missing — {ckpt_path}')
        continue

    print(f'\nFold {fold_idx}: loading checkpoint...')
    model = load_panderm_encoder(str(ckpt_path), CONFIG['nb_classes'], device)

    all_features = []
    with torch.no_grad():
        for batch_imgs, batch_labels, batch_pids in loader:
            batch_imgs = batch_imgs.to(device)
            feats = model.forward_features(batch_imgs, is_train=False)
            if isinstance(feats, (list, tuple)):
                feats = feats[0]
            cls_tokens = feats[:, 0, :].cpu().numpy()  # (B, 1024)
            all_features.append(cls_tokens)

    all_features = np.concatenate(all_features, axis=0)  # (1182, 1024)
    np.save(str(out_path), all_features)
    print(f'Fold {fold_idx}: saved {all_features.shape} → {out_path.name}')

# Save labels, patient IDs, fold assignments once
np.save(str(OUT_DIR / 'milk10k_labels.npy'),      all_labels)
np.save(str(OUT_DIR / 'milk10k_patient_ids.npy'), all_patient_ids)
np.save(str(OUT_DIR / 'milk10k_folds.npy'),       all_folds)
print('\nAll features extracted and saved.')

## Cell 7 — K-fold evaluation (CPU)
Logistic regression on extracted features. For each fold:
train = internal model's train split, test = MILK10k fold test split.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score,
    classification_report, confusion_matrix,
    roc_auc_score, cohen_kappa_score,
)
from sklearn.preprocessing import label_binarize

OUT_DIR     = Path(CONFIG['output_dir'])
CLASS_NAMES = ['Nevus', 'Melanoma Invasive', 'Melanoma in situ']
n_folds     = CONFIG['n_folds']

labels      = np.load(str(OUT_DIR / 'milk10k_labels.npy'))
fold_assign = np.load(str(OUT_DIR / 'milk10k_folds.npy'))

fold_results = []

for fold_idx in range(n_folds):
    feat_path = OUT_DIR / f'milk10k_features_fold{fold_idx}.npy'
    if not feat_path.exists():
        print(f'Fold {fold_idx}: features missing — run Cell 6 first')
        continue

    features   = np.load(str(feat_path))          # (1182, 1024)
    train_mask = fold_assign != fold_idx
    test_mask  = ~train_mask

    X_tr, y_tr = features[train_mask], labels[train_mask]
    X_te, y_te = features[test_mask],  labels[test_mask]

    clf = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced',
                             solver='lbfgs', random_state=42, multi_class='multinomial')
    clf.fit(X_tr, y_tr)
    y_pred  = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)

    y_bin  = label_binarize(y_te, classes=[0, 1, 2])
    report = classification_report(y_te, y_pred, target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    try:    macro_auc = roc_auc_score(y_bin, y_proba, average='macro', multi_class='ovr')
    except: macro_auc = float('nan')

    per_class_auc = {}
    for ci, cls in enumerate(CLASS_NAMES):
        try:    per_class_auc[cls] = roc_auc_score(y_bin[:, ci], y_proba[:, ci])
        except: per_class_auc[cls] = float('nan')

    result = {
        'fold':              fold_idx,
        'balanced_accuracy': balanced_accuracy_score(y_te, y_pred),
        'accuracy':          accuracy_score(y_te, y_pred),
        'macro_auc':         macro_auc,
        'macro_f1':          report['macro avg']['f1-score'],
        'cohen_kappa':       cohen_kappa_score(y_te, y_pred),
        'per_class_auc':     per_class_auc,
        'per_class_f1':      {c: report[c]['f1-score']  for c in CLASS_NAMES},
        'confusion_matrix':  confusion_matrix(y_te, y_pred, labels=[0, 1, 2]),
        'y_test': y_te, 'y_pred': y_pred, 'y_proba': y_proba,
        'n_train': int(train_mask.sum()), 'n_test': int(test_mask.sum()),
        'report_text': classification_report(y_te, y_pred,
                                             target_names=CLASS_NAMES, zero_division=0),
    }
    fold_results.append(result)
    print(f'  Fold {fold_idx+1}: BalAcc={result["balanced_accuracy"]:.3f}  '
          f'AUC={macro_auc:.3f}  F1={result["macro_f1"]:.3f}  '
          f'(train={result["n_train"]}, test={result["n_test"]})')

# Aggregate
scalar_metrics = ['balanced_accuracy', 'accuracy', 'macro_auc', 'macro_f1', 'cohen_kappa']
agg = {}
for m in scalar_metrics:
    vals = [r[m] for r in fold_results]
    agg[f'{m}_mean'] = np.nanmean(vals)
    agg[f'{m}_std']  = np.nanstd(vals)
for cls in CLASS_NAMES:
    for mt in ['f1', 'auc']:
        vals = [r[f'per_class_{mt}'][cls] for r in fold_results]
        agg[f'{cls}_{mt}_mean'] = np.nanmean(vals)
        agg[f'{cls}_{mt}_std']  = np.nanstd(vals)
agg_cm = sum(r['confusion_matrix'] for r in fold_results)

print('\n' + '='*70)
print('RESULTS — MILK10k External Validation (5-Fold CV)')
print('='*70)
print(f"  Balanced Accuracy : {agg['balanced_accuracy_mean']:.3f} +/- {agg['balanced_accuracy_std']:.3f}")
print(f"  Accuracy          : {agg['accuracy_mean']:.3f} +/- {agg['accuracy_std']:.3f}")
print(f"  Macro AUC         : {agg['macro_auc_mean']:.3f} +/- {agg['macro_auc_std']:.3f}")
print(f"  Macro F1          : {agg['macro_f1_mean']:.3f} +/- {agg['macro_f1_std']:.3f}")
print(f"  Cohen Kappa       : {agg['cohen_kappa_mean']:.3f} +/- {agg['cohen_kappa_std']:.3f}")
print(f"\n  {'Class':<22} {'F1':>14} {'AUC':>14}")
print('  ' + '-'*50)
for cls in CLASS_NAMES:
    f1 = f"{agg[f'{cls}_f1_mean']:.3f}+/-{agg[f'{cls}_f1_std']:.3f}"
    au = f"{agg[f'{cls}_auc_mean']:.3f}+/-{agg[f'{cls}_auc_std']:.3f}"
    print(f'  {cls:<22} {f1:>14} {au:>14}')
print('='*70)

# Save results
rows = []
for r in fold_results:
    row = {k: r[k] for k in ['fold','balanced_accuracy','accuracy','macro_auc','macro_f1','cohen_kappa','n_train','n_test']}
    for cls in CLASS_NAMES:
        row[f'{cls}_f1']  = r['per_class_f1'][cls]
        row[f'{cls}_auc'] = r['per_class_auc'][cls]
    rows.append(row)
pd.DataFrame(rows).to_csv(OUT_DIR / 'milk10k_kfold_results.csv', index=False)
print('\nSaved: milk10k_kfold_results.csv')

FOLD_RESULTS = fold_results
AGG          = agg
AGG_CM       = agg_cm

## Cell 8 — Plots

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc as sk_auc
from sklearn.preprocessing import label_binarize

COLORS       = ['#1565c0', '#d32f2f', '#6a1b9a']
DISPLAY_NAMES = {
    'Nevus':              'Nevus',
    'Melanoma Invasive':  'Melanoma Invasive',
    'Melanoma in situ':   'Melanoma In Situ',
}
OUT = Path(CONFIG['output_dir'])

# Confusion matrix
cm_norm = AGG_CM.astype(float) / (AGG_CM.sum(axis=1, keepdims=True) + 1e-9)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(AGG_CM, annot=False, cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
for i in range(AGG_CM.shape[0]):
    for j in range(AGG_CM.shape[1]):
        ax.text(j+0.5, i+0.5, f'{AGG_CM[i,j]}\n({cm_norm[i,j]:.0%})',
                ha='center', va='center', fontsize=10,
                color='white' if cm_norm[i,j] > 0.5 else 'black')
ax.set_title('MILK10k — Aggregate Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(str(OUT / 'milk10k_confusion_matrix.png'), dpi=200, bbox_inches='tight')
plt.show(); print('Saved: milk10k_confusion_matrix.png')

# ROC curves
fig, ax = plt.subplots(figsize=(8, 7))
for ci, cls in enumerate(CLASS_NAMES):
    all_fpr    = np.linspace(0, 1, 100)
    tpr_interps = []
    for r in FOLD_RESULTS:
        y_bin = label_binarize(r['y_test'], classes=[0, 1, 2])
        fpr, tpr, _ = roc_curve(y_bin[:, ci], r['y_proba'][:, ci])
        ti = np.interp(all_fpr, fpr, tpr); ti[0] = 0.0
        tpr_interps.append(ti)
    mean_tpr = np.mean(tpr_interps, axis=0); mean_tpr[-1] = 1.0
    std_tpr  = np.std(tpr_interps, axis=0)
    ax.plot(all_fpr, mean_tpr, color=COLORS[ci], linewidth=2,
            label=f'{DISPLAY_NAMES[cls]} (AUC={sk_auc(all_fpr, mean_tpr):.3f})')
    ax.fill_between(all_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr,
                    color=COLORS[ci], alpha=0.15)
ax.plot([0,1],[0,1],'k--',linewidth=1,alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('MILK10k — Mean ROC Curves (5-Fold CV)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(str(OUT / 'milk10k_roc_curves.png'), dpi=200, bbox_inches='tight')
plt.show(); print('Saved: milk10k_roc_curves.png')

# UMAP
try:
    import umap
    feats_fold0 = np.load(str(OUT / 'milk10k_features_fold0.npy'))
    reducer     = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
    embedding   = reducer.fit_transform(feats_fold0)
    fig, ax     = plt.subplots(figsize=(9, 7))
    for lv, cls in enumerate(CLASS_NAMES):
        mask = labels == lv
        ax.scatter(embedding[mask,0], embedding[mask,1],
                   c=COLORS[lv], s=30, alpha=0.7, edgecolors='k', linewidth=0.3,
                   label=DISPLAY_NAMES[cls])
    ax.set_title('UMAP — MILK10k Features (PanDerm fine-tuned)', fontsize=13, fontweight='bold')
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2'); ax.legend()
    ax.grid(linestyle='--', alpha=0.3); plt.tight_layout()
    plt.savefig(str(OUT / 'milk10k_umap.png'), dpi=200, bbox_inches='tight')
    plt.show(); print('Saved: milk10k_umap.png')
except ImportError:
    print('umap-learn not installed — pip install umap-learn')

print('\nAll plots saved.')